# 05 · Temporal & multimodal alignment

The temporal toolkit that lets per-frame / per-token / per-sample model features meet TR-resolved brain data: `temporal_bin`, `hrf_convolve`, `synchronize_modalities`, and the temporal-shift null.

> Run on EC2 (GPU + Brain-Score data). Do **not** run on a laptop.


In [ ]:
import numpy as np
from brainscore_core.supported_data_standards.brainio.assemblies import NeuroidAssembly
from brainscore_core.temporal import synchronize_modalities, hrf_convolve, double_gamma_hrf

## Two modality streams on different native rates → one TR grid

In [ ]:
def stream(n_pres, src_bins, n_neuroid, scale):
    nb=len(src_bins); data=np.stack([np.full((n_pres,n_neuroid), b*scale) for b in range(nb)],1)
    return NeuroidAssembly(data, dims=['presentation','time_bin','neuroid'],
        coords={'clip_id':('presentation',[f'c{i}' for i in range(n_pres)]),
                'stimulus_id':('presentation',[f'c{i}' for i in range(n_pres)]),
                'time_bin_start_ms':('time_bin',[s for s,_ in src_bins]),
                'time_bin_end_ms':('time_bin',[e for _,e in src_bins]),
                'neuroid_id':('neuroid',np.arange(n_neuroid)),
                'layer':('neuroid',['L']*n_neuroid)})
video = stream(3, [(0,250),(250,500),(500,750),(750,1000)], 4, 1.0)
audio = stream(3, [(0,500),(500,1000)], 6, 10.0)
merged = synchronize_modalities({'video':video,'audio':audio}, [(0,500),(500,1000)])
print('merged shape:', dict(merged.sizes))
print('modalities:', set(merged['modality'].values.tolist()))

## HRF convolution (causal — no future leakage)

In [ ]:
hrf = double_gamma_hrf(duration_sec=30, sampling_rate_hz=1.0)
feats = np.zeros((60, 3)); feats[10] = 1.0
conv = hrf_convolve(feats, sampling_rate_hz=1.0)
print('peak shifted to TR:', int(np.argmax(conv[:,0])), '(impulse at 10, HRF peaks ~5s later)')

## Temporal-shift null — mis-time features, watch prediction degrade

In [ ]:
from brainscore_core.nulls import shift_features, temporal_shift_curve
rng=np.random.RandomState(0); X=rng.randn(240,6); W=rng.randn(6,4); Y=X@W+0.1*rng.randn(240,4)
def r(Xs):
    h=120; from numpy.linalg import solve
    Xtr,Ytr,Xte,Yte=Xs[:h],Y[:h],Xs[h:],Y[h:]
    Wf=solve(Xtr.T@Xtr+np.eye(6), Xtr.T@(Ytr-Ytr.mean(0)))
    p=Xte@Wf+Ytr.mean(0)
    return float(np.mean([np.corrcoef(p[:,j],Yte[:,j])[0,1] for j in range(4)]))
curve=temporal_shift_curve(r, X, [-10,-3,0,3,10])
print('shift -> score:', {k:round(v,2) for k,v in curve.items()})
print('peaks at 0:', curve[0]==max(curve.values()))

**Why this null matters:** if shifting the model features in time does *not* drop the score, the alignment was never carrying stimulus-locked information — the correlation was an artifact.